# NB02c — Weekly Rakuten Snapshot Ingestion
**Signal/Pulse · Temporal Data Layer**

## Purpose

The `products` table captures a single point-in-time snapshot of the Rakuten
catalog. This notebook adds a time-series layer: `products_weekly`, which
accumulates one row per product per weekly scrape run.

Over time this enables signals that no static snapshot can produce:
- **Review velocity** — reviews added per brand per week (leading demand indicator)
- **Ranking movement** — position changes within days, weeks ahead of review counts
- **Price tracking** — promotional calendar reconstruction, price elasticity
- **SKU lifecycle** — new product launches and discontinuations
- **Compound signals** — price × trends × YouTube cross-source triangulation

## Workflow

Run this notebook **after every NB01a scrape run** (weekly).
It is safe to re-run — all ingestion is idempotent via UNIQUE constraints.

**Sections:**
1. Schema — create `products_weekly` table
2. Ingest function — `ingest_rakuten_snapshot`
3. Backfill — apply to all available snapshots
4. Weekly run — apply to new snapshot files
5. Validation and analytical queries
6. Signal preview — velocity, ranking, new products


## 0. Setup

In [1]:
import sys
sys.path.insert(0, "..")

import json
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime, date
from src.schema import get_connection, DB_PATH
from src.utils import DATA_RAW
from src.config import load_rakuten_genres

try:
    conn.execute("SELECT 1")
except Exception:
    conn = get_connection()

RAK_PRODUCTS_DIR = DATA_RAW / "rakuten" / "products"
RAK_RANKING_DIR  = DATA_RAW / "rakuten" / "ranking"

print(f"DB path : {DB_PATH}")
print(f"Products table : "
      f"{conn.execute('SELECT COUNT(*) FROM products WHERE source_id=1').fetchone()[0]:,} rows")
print()

# Show available snapshot files
product_files = sorted(RAK_PRODUCTS_DIR.glob("rakuten_products_*.json"))
dates = sorted(set(
    f.stem.split('_')[3] for f in product_files
    if len(f.stem.split('_')) > 3
))
print(f"Snapshot dates available in raw files: {dates}")


DB path : C:\Users\stanl\OneDrive\Desktop\VSCode_Projects\6. Beauty_ConsumerPulse\data\signal_pulse.db
Products table : 36,255 rows

Snapshot dates available in raw files: ['2026-04-02', '2026-04-10', '2026-04-17', '2026-04-25', '2026-05-05', '2026-05-12']


## 1. Schema — products_weekly

One row per `(product_id, snapshot_date)`. Idempotent — safe to re-run.

**products** = current-state master (always reflects latest values)
**products_weekly** = full history (never overwritten, only appended)

| Column | Description |
|---|---|
| `product_id` | FK to products — the canonical join key |
| `source_item_id` | Rakuten item_code — denormalised for ingest speed |
| `snapshot_date` | YYYY-MM-DD from scrape filename |
| `price_jpy` | Price this week — tracks promotions |
| `review_count` | Cumulative reviews this week — velocity via LAG() |
| `review_avg` | Rating this week — sentiment drift |
| `ranking_position` | Rank in genre this week — leading indicator |
| `is_ranking` | 1 = from ranking API, 0 = from item search |
| `is_new_product` | 1 = first time this product appeared in catalog |


In [2]:
DDL_PRODUCTS_WEEKLY = """
CREATE TABLE IF NOT EXISTS products_weekly (
    snapshot_id      INTEGER PRIMARY KEY,
    product_id       INTEGER NOT NULL,
    source_item_id   TEXT    NOT NULL,
    snapshot_date    TEXT    NOT NULL,
    price_jpy        INTEGER,
    review_count     INTEGER,
    review_avg       REAL,
    ranking_position INTEGER,
    is_ranking       INTEGER DEFAULT 0,
    is_new_product   INTEGER DEFAULT 0,
    ingested_at      TEXT    DEFAULT (datetime('now','utc')),
    UNIQUE(product_id, snapshot_date),
    FOREIGN KEY(product_id) REFERENCES products(product_id)
)
"""

DDL_IDX_PW_DATE    = "CREATE INDEX IF NOT EXISTS idx_pw_date ON products_weekly(snapshot_date)"
DDL_IDX_PW_PRODUCT = "CREATE INDEX IF NOT EXISTS idx_pw_product ON products_weekly(product_id)"
DDL_IDX_PW_NEW     = "CREATE INDEX IF NOT EXISTS idx_pw_new ON products_weekly(is_new_product, snapshot_date)"

conn.execute(DDL_PRODUCTS_WEEKLY)
conn.execute(DDL_IDX_PW_DATE)
conn.execute(DDL_IDX_PW_PRODUCT)
conn.execute(DDL_IDX_PW_NEW)
conn.commit()

print("products_weekly table created ✓")
print("Indexes created ✓")
print()

# Verify schema
print(pd.read_sql("PRAGMA table_info(products_weekly)", conn).to_string(index=False))


products_weekly table created ✓
Indexes created ✓

 cid             name    type  notnull            dflt_value  pk
   0      snapshot_id INTEGER        0                   NaN   1
   1       product_id INTEGER        1                   NaN   0
   2   source_item_id    TEXT        1                   NaN   0
   3    snapshot_date    TEXT        1                   NaN   0
   4        price_jpy INTEGER        0                   NaN   0
   5     review_count INTEGER        0                   NaN   0
   6       review_avg    REAL        0                   NaN   0
   7 ranking_position INTEGER        0                   NaN   0
   8       is_ranking INTEGER        0                     0   0
   9   is_new_product INTEGER        0                     0   0
  10      ingested_at    TEXT        0 datetime('now','utc')   0


## 2. Ingest Function — ingest_rakuten_snapshot

Single function that handles both tables atomically:
1. **Upserts** into `products` — current state always up to date
2. **Inserts** into `products_weekly` — history preserved
3. **Flags** new products — `is_new_product=1` on first appearance

Idempotent: re-running the same snapshot file produces zero new rows
due to UNIQUE constraint on `(product_id, snapshot_date)`.


In [3]:
def ingest_rakuten_snapshot(
    conn,
    item: dict,
    category_id: int,
    snapshot_date: str,
    ranking_position: int = None,
    is_ranking: bool = False,
) -> dict:
    """
    Ingest one Rakuten product into both products (current state)
    and products_weekly (history).

    Returns dict with keys: inserted_product, inserted_weekly, is_new

    Design notes:
    - products UPDATE uses COALESCE — a NULL from a failed scrape never
      overwrites a previously good value.
    - products_weekly uses ON CONFLICT upsert — if the same product appears
      in both item-search and ranking files for the same date, the ranking
      file wins on ranking_position and is_ranking (more valuable signal).
    """
    source_item_id = (
        item.get('itemCode') or
        item.get('item_code') or
        item.get('itemUrl', '')
    )
    if not source_item_id:
        return {'inserted_product': False, 'inserted_weekly': False, 'is_new': False}

    product_name   = item.get('itemName') or item.get('item_name', '')
    price_jpy      = item.get('itemPrice') or item.get('price_jpy')
    review_count   = item.get('reviewCount') or item.get('review_count', 0)
    review_avg     = item.get('reviewAverage') or item.get('review_avg')

    try:
        price_jpy    = int(price_jpy)    if price_jpy    is not None else None
        review_count = int(review_count) if review_count is not None else 0
        review_avg   = float(review_avg) if review_avg   is not None else None
        if ranking_position:
            ranking_position = int(ranking_position)
    except (ValueError, TypeError):
        pass

    cur = conn.cursor()

    # ── Step 1: Check if product already exists ────────────────────────────
    existing = cur.execute(
        "SELECT product_id FROM products "
        "WHERE source_id = 1 AND source_item_id = ?",
        (source_item_id,)
    ).fetchone()

    is_new = existing is None

    # ── Step 2: Upsert into products (current state) ───────────────────────
    if is_new:
        cur.execute("""
            INSERT OR IGNORE INTO products
                (source_id, source_item_id, product_name, category_id,
                 price_jpy, review_count, review_avg,
                 ranking_position, snapshot_date)
            VALUES (1, ?, ?, ?, ?, ?, ?, ?, ?)
        """, (
            source_item_id, product_name, category_id,
            price_jpy, review_count, review_avg,
            ranking_position, snapshot_date
        ))
        product_id = cur.lastrowid
        inserted_product = cur.rowcount > 0
    else:
        product_id = existing[0]
        # COALESCE: a NULL value from a failed scrape never overwrites
        # a previously good value in the master products table.
        cur.execute("""
            UPDATE products SET
                review_count     = COALESCE(?, review_count),
                review_avg       = COALESCE(?, review_avg),
                price_jpy        = COALESCE(?, price_jpy),
                ranking_position = COALESCE(?, ranking_position),
                snapshot_date    = ?
            WHERE product_id = ?
        """, (
            review_count, review_avg, price_jpy,
            ranking_position, snapshot_date,
            product_id
        ))
        inserted_product = False

    # ── Step 3: Upsert into products_weekly (history) ─────────────────────
    # ON CONFLICT: if the product already has a row for this date (from the
    # item-search file), and we now see it in the ranking file, promote the
    # ranking data. Item-search data is never downgraded by this update.
    cur.execute("""
        INSERT INTO products_weekly
            (product_id, source_item_id, snapshot_date,
             price_jpy, review_count, review_avg,
             ranking_position, is_ranking, is_new_product)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
        ON CONFLICT(product_id, snapshot_date) DO UPDATE SET
            ranking_position = CASE WHEN excluded.is_ranking = 1
                                    THEN excluded.ranking_position
                                    ELSE ranking_position END,
            is_ranking       = CASE WHEN excluded.is_ranking = 1
                                    THEN 1
                                    ELSE is_ranking END
    """, (
        product_id, source_item_id, snapshot_date,
        price_jpy, review_count, review_avg,
        ranking_position, int(is_ranking), int(is_new)
    ))
    inserted_weekly = cur.rowcount > 0

    return {
        'inserted_product': inserted_product,
        'inserted_weekly':  inserted_weekly,
        'is_new':           is_new,
    }


print("ingest_rakuten_snapshot() defined ✓")
print()
print("Behaviour:")
print("  New product      → INSERT products + INSERT products_weekly (is_new=1)")
print("  Existing product → UPDATE products (COALESCE) + upsert products_weekly")
print("  Re-run same date → ON CONFLICT promotes ranking data if present,")
print("                     otherwise no-op (idempotent)")


ingest_rakuten_snapshot() defined ✓

Behaviour:
  New product      → INSERT products + INSERT products_weekly (is_new=1)
  Existing product → UPDATE products (COALESCE) + upsert products_weekly
  Re-run same date → ON CONFLICT promotes ranking data if present,
                     otherwise no-op (idempotent)


## 3. Backfill — Apply to All Available Snapshots

`products_weekly` holds the full snapshot history. This section backfills
it from every Rakuten snapshot present in `data/raw/rakuten/` — the
available dates are detected in Setup as `dates`.

Idempotent: the `UNIQUE(product_id, snapshot_date)` constraint means
re-running produces zero new rows for snapshots already ingested.


In [4]:
# ── Load genre → category_id mapping ─────────────────────────────────────
genres = load_rakuten_genres(layer1=True)

# default_cat defined unconditionally — used as fallback for any genre
# that doesn't resolve in the categories table.
default_cat = None
rak_cat_ids = {}

for _, row in genres.iterrows():
    genre_id = str(row['rakuten_genre_id'])
    cat_row  = conn.execute("""
        SELECT category_id FROM categories
        WHERE source_id = 1
          AND normalized_name LIKE ?
        LIMIT 1
    """, (f"%{genre_id}%",)).fetchone()
    if cat_row:
        rak_cat_ids[genre_id] = cat_row[0]

# Fallback: use any Rakuten category if mapping is empty
if not rak_cat_ids:
    cats = conn.execute("""
        SELECT DISTINCT category_id FROM products
        WHERE source_id = 1 LIMIT 1
    """).fetchone()
    default_cat = cats[0] if cats else None
    print(f"WARNING: genre mapping empty — using fallback category_id: {default_cat}")
else:
    # Set default_cat to first resolved value for any unmapped genres
    default_cat = next(iter(rak_cat_ids.values()))
    print(f"Genre → category_id mapping: {len(rak_cat_ids)} genres resolved")
    print(f"Fallback category_id for unmapped genres: {default_cat}")


def run_snapshot_ingest(snapshot_date_filter: str = None) -> dict:
    """
    Ingest all product + ranking files for a given snapshot date
    (or all dates if None) into products_weekly.

    Returns summary dict.
    """
    product_files = sorted(RAK_PRODUCTS_DIR.glob("rakuten_products_*.json"))
    ranking_files = sorted(RAK_RANKING_DIR.glob("rakuten_ranking_*.json"))

    if snapshot_date_filter:
        product_files = [f for f in product_files
                         if snapshot_date_filter in f.stem]
        ranking_files = [f for f in ranking_files
                         if snapshot_date_filter in f.stem]

    totals = {
        'files':            0,
        'weekly_inserted':  0,
        'weekly_skipped':   0,
        'new_products':     0,
        'products_updated': 0,
    }

    # Product files first, ranking files second.
    # ON CONFLICT in ingest_rakuten_snapshot promotes ranking data
    # when the same product appears in both.
    all_files = (
        [(f, False) for f in product_files] +
        [(f, True)  for f in ranking_files]
    )

    for fpath, is_ranking_file in all_files:
        parts         = fpath.stem.split('_')
        genre_id      = parts[2]
        snapshot_date = parts[3] if len(parts) > 3 else str(date.today())
        cat_id        = rak_cat_ids.get(genre_id, default_cat)

        with open(fpath, encoding='utf-8') as f:
            data = json.load(f)

        items = (
            data if isinstance(data, list)
            else data.get('items', data.get('products', []))
        )

        if not items:
            continue

        totals['files'] += 1
        file_inserted = 0

        for i, wrapper in enumerate(items):
            item   = wrapper.get('Item', wrapper)
            result = ingest_rakuten_snapshot(
                conn, item,
                category_id=cat_id,
                snapshot_date=snapshot_date,
                ranking_position=i + 1,
                is_ranking=is_ranking_file,
            )
            if result['inserted_weekly']:
                totals['weekly_inserted'] += 1
                file_inserted += 1
            else:
                totals['weekly_skipped'] += 1
            if result['is_new']:
                totals['new_products'] += 1
            if not result['inserted_product'] and not result['is_new']:
                totals['products_updated'] += 1

        conn.commit()
        print(f"  {fpath.name:<55} {file_inserted:>5} rows")

    return totals


print()
# Backfill products_weekly from EVERY available snapshot — rebuilds the
# full history from a fresh database. `dates` (all available snapshot
# dates) is computed in the Setup cell.
print(f"Backfilling products_weekly from {len(dates)} snapshots: {dates}")
print("=" * 65)
grand = {'files': 0, 'weekly_inserted': 0, 'weekly_skipped': 0}
for snap in dates:
    totals = run_snapshot_ingest(snapshot_date_filter=snap)
    for k in grand:
        grand[k] += totals[k]
    print(f"  {snap}: {totals['files']} files, "
          f"{totals['weekly_inserted']:,} weekly rows added")
print()
print("BACKFILL COMPLETE")
print(f"  Snapshots processed : {len(dates)}")
print(f"  Files processed     : {grand['files']:,}")
print(f"  Weekly rows added   : {grand['weekly_inserted']:,}")
print(f"  Skipped (duplicate) : {grand['weekly_skipped']:,}")



Backfilling products_weekly from 6 snapshots: ['2026-04-02', '2026-04-10', '2026-04-17', '2026-04-25', '2026-05-05', '2026-05-12']


  rakuten_products_100939_2026-04-02.json                  3000 rows


  rakuten_products_100944_2026-04-02.json                  3000 rows


  rakuten_products_204233_2026-04-02.json                  3000 rows


  rakuten_products_216301_2026-04-02.json                  3000 rows


...

  rakuten_products_216387_2026-05-12.json                  3000 rows


  rakuten_products_216424_2026-05-12.json                  3000 rows


  rakuten_products_216492_2026-05-12.json                  3000 rows


  rakuten_products_563728_2026-05-12.json                  3000 rows


  rakuten_products_564517_2026-05-12.json                  3000 rows
  rakuten_ranking_100939_2026-05-12.json                     30 rows
  rakuten_ranking_100944_2026-05-12.json                     30 rows
  rakuten_ranking_204233_2026-05-12.json                     29 rows
  rakuten_ranking_216301_2026-05-12.json                     30 rows
  rakuten_ranking_216307_2026-05-12.json                     30 rows
  rakuten_ranking_216348_2026-05-12.json                     30 rows
  rakuten_ranking_216387_2026-05-12.json                     30 rows
  rakuten_ranking_216424_2026-05-12.json                     30 rows
  rakuten_ranking_216492_2026-05-12.json                     30 rows
  rakuten_ranking_563728_2026-05-12.json                     30 rows
  rakuten_ranking_564517_2026-05-12.json                     29 rows
  2026-05-12: 22 files, 33,328 weekly rows added

BACKFILL COMPLETE
  Snapshots processed : 6
  Files processed     : 132
  Weekly rows added   : 199,977
  Skipped (duplica

## 4. Weekly Run — Apply to New Snapshot Files

Run this section every week after NB01a completes.
Change `SNAPSHOT_DATE` to match the date in your new filenames.

The cell is idempotent — re-running the same date produces zero new rows.


In [5]:
# ── Set to today's scrape date ───────────────────────────────────────────
SNAPSHOT_DATE = str(date.today())  # e.g. '2026-04-06'

# Check files exist for this date
files_today  = list(RAK_PRODUCTS_DIR.glob(f"rakuten_products_*_{SNAPSHOT_DATE}.json"))
files_today += list(RAK_RANKING_DIR.glob(f"rakuten_ranking_*_{SNAPSHOT_DATE}.json"))

if not files_today:
    print(f"No files found for {SNAPSHOT_DATE}.")
    print("Run NB01a first, then re-run this cell.")
else:
    # Empty-file check — handles both list and dict JSON structures
    empty = []
    for fpath in files_today:
        data  = json.loads(fpath.read_text(encoding='utf-8'))
        items = (
            data if isinstance(data, list)
            else data.get('items', data.get('products', []))
        )
        if not items:
            empty.append(fpath.name)

    if empty:
        print(f"WARNING: {len(empty)} empty files found — possible API errors.")
        print("Delete these before ingesting:")
        for f in empty:
            print(f"  {f}")
        print("\nAborted. Fix empty files then re-run.")
    else:
        print(f"Running weekly ingest for {SNAPSHOT_DATE}...")
        print(f"{len(files_today)} files found")
        print("=" * 65)
        totals = run_snapshot_ingest(snapshot_date_filter=SNAPSHOT_DATE)
        print()
        print("=" * 65)
        print(f"WEEKLY INGEST COMPLETE — {SNAPSHOT_DATE}")
        print(f"  Files processed    : {totals['files']:,}")
        print(f"  Weekly rows added  : {totals['weekly_inserted']:,}")
        print(f"  Skipped (duplicate): {totals['weekly_skipped']:,}")
        print(f"  New products       : {totals['new_products']:,}")
        print(f"  Products updated   : {totals['products_updated']:,}")


No files found for 2026-05-17.
Run NB01a first, then re-run this cell.


## 5. Validation

Confirm the table is populated correctly and the time series is intact.


In [6]:
# ── Table overview ────────────────────────────────────────────────────────
print("products_weekly — overview:")
print(pd.read_sql("""
    SELECT
        snapshot_date,
        COUNT(*)                    AS total_rows,
        SUM(is_new_product)         AS new_products,
        SUM(is_ranking)             AS from_ranking_api,
        COUNT(DISTINCT product_id)  AS unique_products
    FROM products_weekly
    GROUP BY snapshot_date
    ORDER BY snapshot_date
""", conn).to_string(index=False))

print()

# ── Products table current state ──────────────────────────────────────────
print("products table — snapshot_date distribution after update:")
print(pd.read_sql("""
    SELECT snapshot_date, COUNT(*) as products
    FROM products
    WHERE source_id = 1
    GROUP BY snapshot_date
    ORDER BY snapshot_date
""", conn).to_string(index=False))

print()

# ── Integrity check ───────────────────────────────────────────────────────
orphans = conn.execute("""
    SELECT COUNT(*) FROM products_weekly pw
    LEFT JOIN products p ON pw.product_id = p.product_id
    WHERE p.product_id IS NULL
""").fetchone()[0]
print(f"Orphaned weekly rows (no matching product): {orphans} "
      f"{"✓" if orphans == 0 else "✗ CHECK REQUIRED"}")

print()

# ── Convenience view — eliminates repeated 3-table join in signal queries ─
# tier_canonical = COALESCE(tier_predicted, tier_override, c.tier)
# product_name   denormalised for display without joining products each time
conn.execute("""
    CREATE VIEW IF NOT EXISTS products_weekly_full AS
    SELECT
        pw.*,
        COALESCE(p.tier_predicted, p.tier_override, c.tier) AS tier_canonical,
        p.product_name
    FROM products_weekly pw
    JOIN products   p ON pw.product_id  = p.product_id
    JOIN categories c ON p.category_id  = c.category_id
""")
conn.commit()
print("View products_weekly_full created ✓")
print("  Columns: all products_weekly cols + tier_canonical + product_name")
print("  Use instead of manual 3-table joins in signal queries.")


products_weekly — overview:


snapshot_date  total_rows  new_products  from_ranking_api  unique_products
   2026-04-02       28949            48               284            28949
   2026-04-10       29143            16               286            29143
   2026-04-17       29190             9               290            29190
   2026-04-25       28811            29               292            28811
   2026-05-05       29104            23               292            29104
   2026-05-12       29200             6               285            29200

products table — snapshot_date distribution after update:


snapshot_date  products
   2026-04-02       815
   2026-04-10       886
   2026-04-17       996
   2026-04-25      1279
   2026-05-05      2391
   2026-05-12     29200
   2026-05-17       819

Orphaned weekly rows (no matching product): 0 ✓

View products_weekly_full created ✓
  Columns: all products_weekly cols + tier_canonical + product_name
  Use instead of manual 3-table joins in signal queries.


## 6. Signal Preview

These queries are the foundation of the NB07 dashboard.
With one snapshot they show current state.
With multiple snapshots they show velocity, momentum, and change.

**Note:** LAG()-based signals require at least 2 snapshots.
Run NB01a → NB02c again next week and these queries come alive.


In [7]:
# ── Signal 1: New products this snapshot ─────────────────────────────────
# Products appearing in Rakuten for the first time.
# With weekly cadence: what launched this week?

latest_date = conn.execute("""
    SELECT MAX(snapshot_date) FROM products_weekly
""").fetchone()[0]

print(f"NEW PRODUCTS — {latest_date}")
print("=" * 65)
print(pd.read_sql("""
    SELECT
        SUBSTR(product_name, 1, 55)  AS product_name,
        price_jpy,
        review_count,
        tier_canonical               AS tier
    FROM products_weekly_full
    WHERE is_new_product = 1
      AND snapshot_date  = ?
    ORDER BY review_count DESC
    LIMIT 20
""", conn, params=(latest_date,)).to_string(index=False))


NEW PRODUCTS — 2026-05-12
                                           product_name  price_jpy  review_count     tier
【最大35％OFFクーポン配布中 5/16 1:59まで】 公式 ビオルチア Bio Lucia まつ毛美容液       3980          2367 skincare
【神トク20%OFFクーポン★5/9 20時開始】【神トク限定セット】 トゥヴェール ベストセラー スキンケア       8730            29 skincare
    ビオレ おうちdeエステ ディープクレイ洗顔 リフレッシュアロマの香り 180g 洗顔料 アットコスメ       1168             7 skincare
神トククーポンで＼49％OFF＋ギフト／【VT公式】【楽天限定】【ビタつや肌セット】ビタミンC アスタキサンチ       7700             0 skincare
 400円クーポン配布中【国内正規品】(リニューアル) ナリス マジェスタ ミルク 80ml×1本・3本セット       6180             0 skincare
★24時間限定最大45％OFF！★【日本公式販売店】Torriden トリデン 24時間限定特価イベント 送料       2970             0 skincare


In [8]:
# ── Signal 2: Review velocity ────────────────────────────────────────────
# Reviews added per product per snapshot period.
# With 1 snapshot: shows current review count as baseline.
# With 2+ snapshots: shows week-on-week acceleration.

snapshots = conn.execute("""
    SELECT COUNT(DISTINCT snapshot_date) FROM products_weekly
""").fetchone()[0]

if snapshots < 2:
    print("Review velocity requires 2+ snapshots.")
    print(f"Current snapshots: {snapshots}")
    print("Run NB01a → NB02c next week to unlock this signal.")
    print()
    print("BASELINE — Top products by review count (current state):")
    print(pd.read_sql("""
        SELECT
            SUBSTR(product_name, 1, 50) AS product_name,
            review_count,
            tier_canonical              AS tier
        FROM products_weekly_full
        WHERE review_count IS NOT NULL
        ORDER BY review_count DESC
        LIMIT 15
    """, conn).to_string(index=False))
else:
    print("REVIEW VELOCITY — Week-on-week change:")
    print(pd.read_sql("""
        WITH weekly_delta AS (
            SELECT
                product_id,
                snapshot_date,
                review_count,
                review_count - LAG(review_count) OVER (
                    PARTITION BY product_id
                    ORDER BY snapshot_date
                ) AS reviews_added
            FROM products_weekly
        )
        SELECT
            SUBSTR(pwf.product_name, 1, 50)  AS product_name,
            wd.snapshot_date,
            wd.review_count,
            wd.reviews_added,
            pwf.tier_canonical               AS tier
        FROM weekly_delta wd
        JOIN products_weekly_full pwf
          ON wd.product_id = pwf.product_id
         AND wd.snapshot_date = pwf.snapshot_date
        WHERE wd.reviews_added IS NOT NULL
          AND wd.snapshot_date = (SELECT MAX(snapshot_date) FROM products_weekly)
        ORDER BY wd.reviews_added DESC
        LIMIT 20
    """, conn).to_string(index=False))


REVIEW VELOCITY — Week-on-week change:


                                      product_name snapshot_date  review_count  reviews_added      tier
スキンクリア クレンズ オイル エコパック*大容量 (4種) [ クレンジング クレンジングオイル     2026-05-12         45762            181  skincare
マイルドクレンジング オイル【ファンケル 公式】 [FANCL クレンジング マイクレ 無添加 毛穴    2026-05-12         14242            136  skincare
【LINEお友だち追加で500円クーポン】白髪染め シャンプー トリートメント KUROクリームシャ    2026-05-12          7229            106  skincare
スキンクリア クレンズ オイル (レギュラーボトル) (セット/単品) [ クレンジング クレンジン    2026-05-12         38760             94  skincare
⭐️楽天1位・4冠⭐️ペプチドショットアンプル2X 美容液 30ml 100ml アンプルエヌ ペプ    2026-05-12           545             89  skincare
【予約販売】【レビュー投稿で次回50%OFF】【公式】マグバーム 無添加 お試し用 トラベル用 リポ    2026-05-12           896             79  skincare
数量限定・新登場【ポイント10倍】 スキンクリア クレンズ オイル スムースブラック クレンジング     2026-05-12           286             75  skincare
【正規販売店/楽天1位】ロアザオイル LOA THE OIL 30ml / 100ml ブランシュ     2026-05-12          3119             72     other
【公式】Yunth 生ビタミンC 美白美容液 1ml×28包 | 美容液 ビタミンC 導入美容液 先    2026-05-12

In [9]:
# ── Signal 3: Ranking movement ───────────────────────────────────────────
# Position change vs previous snapshot.
# Negative delta = moved UP (improvement). Positive delta = moved DOWN.

if snapshots < 2:
    print("Ranking movement requires 2+ snapshots.")
    print(f"Current snapshots: {snapshots}")
    print("Run NB01a → NB02c next week to unlock this signal.")
    print()
    print("BASELINE — Current ranking positions (top 20):")
    print(pd.read_sql("""
        SELECT
            SUBSTR(product_name, 1, 50) AS product_name,
            ranking_position,
            review_count,
            tier_canonical              AS tier
        FROM products_weekly_full
        WHERE ranking_position IS NOT NULL
          AND is_ranking = 1
        ORDER BY ranking_position ASC
        LIMIT 20
    """, conn).to_string(index=False))
else:
    print("RANKING MOVEMENT — Biggest movers this week:")
    print(pd.read_sql("""
        WITH ranking_delta AS (
            SELECT
                product_id,
                snapshot_date,
                ranking_position,
                ranking_position - LAG(ranking_position) OVER (
                    PARTITION BY product_id ORDER BY snapshot_date
                ) AS rank_delta
            FROM products_weekly
            WHERE is_ranking = 1
        )
        SELECT
            SUBSTR(pwf.product_name, 1, 50)  AS product_name,
            rd.ranking_position              AS rank_now,
            rd.rank_delta,
            CASE WHEN rd.rank_delta < 0
                 THEN '▲ ' || ABS(rd.rank_delta)
                 ELSE '▼ ' || rd.rank_delta
            END                              AS movement,
            pwf.tier_canonical               AS tier
        FROM ranking_delta rd
        JOIN products_weekly_full pwf
          ON rd.product_id   = pwf.product_id
         AND rd.snapshot_date = pwf.snapshot_date
        WHERE rd.rank_delta IS NOT NULL
          AND rd.snapshot_date = (SELECT MAX(snapshot_date) FROM products_weekly)
        ORDER BY ABS(rd.rank_delta) DESC
        LIMIT 20
    """, conn).to_string(index=False))


RANKING MOVEMENT — Biggest movers this week:
                                      product_name  rank_now  rank_delta movement      tier
【なくなり次第終了｜母の日限定BOX】アスタリフト オプミー ジェル 60g（全2種） [医薬部外品         2         -28     ▲ 28  skincare
レチノパワー リンクルクリーム ba S(15g)【エリクシール(ELIXIR)】[レチノール クリ         2         -27     ▲ 27  skincare
＼2,200円OFFクーポン／【楽天1位獲得】ラッシュアディクトアドバンス まつ毛美容液 Lasha         5         -25     ▲ 25 cosmetics
KISO CARE アゼライン酸 15％配合フェイスクリーム キソ バランシングクリームAZ 20g         7         -23     ▲ 23  skincare
★資生堂正規取扱店　エリクシールデーケアレボリューショントーンアップSP＋aa(ベビーピンク)SPF         1         -22     ▲ 22  skincare
【 d'Alba ( ダルバ ) 公式 】【 ビタ トーニング カプセル セラム 50ml/100m        23          21     ▼ 21 cosmetics
＜今なら19800円相当！＞ラヴィットで紹介 ELEKI LIFT【公式】美顔器 リフトアップ ブラ         5         -21     ▲ 21  skincare
【 d'Alba ( ダルバ ) 公式 】【 選べる ウォータフル 日焼け止め 7種 SPF50+         29          20     ▼ 20 cosmetics
＼楽天ランキング1位／【Anua公式】【ビタミン10 PORESTRIX セラム 20ml】ビタミン        22          20     ▼ 20  skincare
★資生堂正規取引店　資生堂 エリクシール デーケアレボリューショントー

In [10]:
# ── Signal 4: Price changes ───────────────────────────────────────────────
# Products whose price changed vs previous snapshot.

SKIN_TIERS = (
    "'skincare','mass_skincare','sensitive_skincare',"
    "'prestige_skincare','sun_protection','dermo_skincare'"
)
ALL_TIERS = SKIN_TIERS + (
    ",'cosmetics','mass_cosmetics',"
    "'prestige_cosmetics','base_makeup'"
)

if snapshots < 2:
    print("Price change detection requires 2+ snapshots.")
    print(f"Current snapshots: {snapshots}")
    print("Run NB01a → NB02c next week to unlock this signal.")
    print()
    print("BASELINE — Price distribution by tier:")
    print(pd.read_sql(f"""
        SELECT
            tier_canonical              AS tier,
            COUNT(*)                    AS products,
            ROUND(AVG(price_jpy), 0)    AS avg_price,
            MIN(price_jpy)              AS min_price,
            MAX(price_jpy)              AS max_price
        FROM products_weekly_full
        WHERE price_jpy IS NOT NULL
          AND tier_canonical IN ({ALL_TIERS})
        GROUP BY tier_canonical
        ORDER BY avg_price DESC
    """, conn).to_string(index=False))
else:
    print("PRICE CHANGES — Products repriced this week:")
    print(pd.read_sql("""
        WITH price_delta AS (
            SELECT
                product_id,
                snapshot_date,
                price_jpy,
                LAG(price_jpy) OVER (
                    PARTITION BY product_id ORDER BY snapshot_date
                ) AS prev_price
            FROM products_weekly
        )
        SELECT
            SUBSTR(pwf.product_name, 1, 50)        AS product_name,
            pd.prev_price                          AS price_before,
            pd.price_jpy                           AS price_now,
            pd.price_jpy - pd.prev_price           AS price_delta,
            ROUND(100.0 * (pd.price_jpy - pd.prev_price)
                  / pd.prev_price, 1)              AS pct_change,
            pwf.tier_canonical                     AS tier
        FROM price_delta pd
        JOIN products_weekly_full pwf
          ON pd.product_id   = pwf.product_id
         AND pd.snapshot_date = pwf.snapshot_date
        WHERE pd.prev_price IS NOT NULL
          AND pd.price_jpy != pd.prev_price
          AND pd.snapshot_date = (SELECT MAX(snapshot_date) FROM products_weekly)
        ORDER BY ABS(pd.price_jpy - pd.prev_price) DESC
        LIMIT 20
    """, conn).to_string(index=False))


PRICE CHANGES — Products repriced this week:


                                      product_name  price_before  price_now  price_delta  pct_change      tier
＜今なら19800円相当！＞ラヴィットで紹介 ELEKI LIFT【公式】美顔器 リフトアップ ブラ         58000      28285       -29715       -51.2  skincare
＼送料無料／【LBクリーム】30g ベルリッチ化粧品 クリーム 保湿 植物幹細胞 細胞成長因子 AC         22000       2980       -19020       -86.5  skincare
スノボ・スキー・ゴルフ・海・釣り・スポーツ、アウトドア、キャンプ【P5倍】レビュー投稿でプレゼント【          3888      19440        15552       400.0  skincare
霊芝 冬虫夏草 SPAクリーム スパクリーム 大容量 詰め替え用 500g (6-7ヶ月分) ハリと         28913      14456       -14457       -50.0  skincare
送料無料！【プレミアジェリー】60g 美容ジェル ヒト幹細胞培養エキス ヒト幹細胞 乾燥 ヒアルロン         16500       2480       -14020       -85.0  skincare
送料無料！【エンジェルスキンセラム 】 20g ベルリッチ化粧品 エイジングケア 美容液 EGF グ         16500       2490       -14010       -84.9  skincare
【期間限定65%OFF! 通常価格19980円税込⇒6980円税込】美顔器 ギフト セット イオンプ          6980      19980        13000       186.2  skincare
エミューの雫 オールインワンゲル 3本セット【ジェル/ゲル/美白/エミューオイル/ザクロ/プロテオグ         17820       7480       -10340       -58.0  skincare
【

## 7. Plotly Signal Dashboard

Interactive charts built directly from `products_weekly_full`.
Each sub-section is self-contained — charts render inline via `fig.show()`.

**Sub-sections:**
- 7.1 Bump chart — ranking trajectory across snapshots
- 7.2 Horizontal bar — review velocity (fastest-accumulating products)
- 7.3 Scatter — price movement vs consumer engagement
- 7.4 Bar — new product arrivals per snapshot date

In [11]:
import plotly.graph_objects as go
import plotly.express as px

# ── Shared tier colour map ─────────────────────────────────────────────────
TIER_COLOURS = {
    'skincare':  '#4A90B8',
    'cosmetics': '#C4627A',
}
DEFAULT_COLOUR = '#78909C'

def tier_colour(tier):
    return TIER_COLOURS.get(tier, DEFAULT_COLOUR)

LAYOUT_DEFAULTS = dict(
    paper_bgcolor='white',
    plot_bgcolor='#FAFAFA',
    font_family='sans-serif',
)

# ── Analytical scope — skincare + cosmetics tiers only ─────────────────────
SKIN_COS_TIERS = (
    "'skincare','mass_skincare','sensitive_skincare',"
    "'prestige_skincare','sun_protection','dermo_skincare',"
    "'cosmetics','mass_cosmetics','prestige_cosmetics','base_makeup'"
)

# ── Snapshot dates (dynamic — never hardcoded) ─────────────────────────────
snapshot_dates = pd.read_sql(
    "SELECT DISTINCT snapshot_date FROM products_weekly ORDER BY snapshot_date",
    conn
)['snapshot_date'].tolist()

print(f"Snapshot dates found: {snapshot_dates}")

# ──────────────────────────────────────────────────────────────────────────
# ## 7.1 Ranking Trajectory (Bump Chart)
# ──────────────────────────────────────────────────────────────────────────
df_rank = pd.read_sql(f"""
    SELECT product_id, product_name, snapshot_date, ranking_position, tier_canonical
    FROM products_weekly_full
    WHERE is_ranking = 1
      AND ranking_position IS NOT NULL
      AND tier_canonical IN ({SKIN_COS_TIERS})
""", conn)

if df_rank.empty or df_rank['snapshot_date'].nunique() < 2:
    print("WARNING 7.1: Not enough ranking snapshots for bump chart — skipping.")
else:
    date_counts = df_rank.groupby('product_id')['snapshot_date'].nunique()
    valid_ids = date_counts[date_counts >= 2].index
    df_rank = df_rank[df_rank['product_id'].isin(valid_ids)]

    if df_rank.empty:
        print("WARNING 7.1: No products appear in 2+ snapshots yet — skipping.")
    else:
        avg_rank = (
            df_rank.groupby('product_id')['ranking_position']
            .mean()
            .nsmallest(20)
        )
        top20_ids = avg_rank.index.tolist()
        df_bump = df_rank[df_rank['product_id'].isin(top20_ids)].copy()

        labels = (
            df_bump[['product_id', 'product_name']]
            .drop_duplicates('product_id')
            .set_index('product_id')['product_name']
            .str[:20]
        )

        fig = go.Figure()
        for pid in top20_ids:
            sub = df_bump[df_bump['product_id'] == pid].sort_values('snapshot_date')
            tier = sub['tier_canonical'].iloc[0]
            colour = tier_colour(tier)
            fig.add_trace(go.Scatter(
                x=sub['snapshot_date'],
                y=sub['ranking_position'],
                mode='lines+markers',
                name=labels.get(pid, str(pid)),
                line=dict(color=colour, width=2),
                marker=dict(size=7),
                hovertemplate=(
                    f"<b>{labels.get(pid, str(pid))}</b><br>"
                    "Date: %{x}<br>Rank: %{y}<extra></extra>"
                ),
            ))

        fig.update_layout(**LAYOUT_DEFAULTS, title="Ranking Trajectory — Top 20 Products Across Snapshots", height=600, legend=dict(font=dict(size=10)))
        fig.update_xaxes(showgrid=False, title="Snapshot date")
        fig.update_yaxes(autorange="reversed", gridcolor='#E0E0E0', title="Ranking position")
        fig.show()
        print("Chart 1 ✓")

# ──────────────────────────────────────────────────────────────────────────
# ## 7.2 Review Velocity (Horizontal Bar)
# ──────────────────────────────────────────────────────────────────────────
df_vel = pd.read_sql(f"""
    SELECT
        product_id,
        MAX(product_name)    AS product_name,
        MAX(tier_canonical)  AS tier_canonical,
        MAX(review_count) - MIN(review_count) AS reviews_added
    FROM products_weekly_full
    WHERE review_count IS NOT NULL
      AND tier_canonical IN ({SKIN_COS_TIERS})
    GROUP BY product_id
    HAVING reviews_added > 0
""", conn)

if df_vel.empty:
    print("WARNING 7.2: No review velocity data — skipping.")
else:
    top15 = df_vel.nlargest(15, 'reviews_added').sort_values('reviews_added', ascending=True)
    top15['label'] = top15['product_name'].str[:35]
    top15['colour'] = top15['tier_canonical'].apply(tier_colour)

    fig = go.Figure(go.Bar(
        x=top15['reviews_added'],
        y=top15['label'],
        orientation='h',
        marker_color=top15['colour'].tolist(),
        text=top15['reviews_added'],
        textposition='outside',
        hovertemplate="<b>%{y}</b><br>Reviews added: %{x}<extra></extra>",
    ))
    fig.update_layout(**LAYOUT_DEFAULTS, title="Review Velocity — Top 15 Products", height=500)
    fig.update_xaxes(showgrid=False, title="Total reviews added across all snapshots")
    fig.update_yaxes(gridcolor='#E0E0E0', automargin=True)
    fig.show()
    print("Chart 2 ✓")

# ──────────────────────────────────────────────────────────────────────────
# ## 7.3 Price Movement vs Consumer Engagement (Scatter)
# ──────────────────────────────────────────────────────────────────────────
df_price = pd.read_sql("""
    WITH first_last AS (
        SELECT
            product_id,
            MIN(snapshot_date) AS first_date,
            MAX(snapshot_date) AS last_date
        FROM products_weekly
        WHERE price_jpy IS NOT NULL
        GROUP BY product_id
        HAVING first_date != last_date
    ),
    prices AS (
        SELECT
            fl.product_id,
            pw_first.price_jpy  AS price_first,
            pw_last.price_jpy   AS price_last,
            pw_last.review_count
        FROM first_last fl
        JOIN products_weekly pw_first
          ON fl.product_id = pw_first.product_id
         AND fl.first_date  = pw_first.snapshot_date
        JOIN products_weekly pw_last
          ON fl.product_id = pw_last.product_id
         AND fl.last_date   = pw_last.snapshot_date
        WHERE pw_first.price_jpy > 0
          AND pw_last.price_jpy  IS NOT NULL
          AND pw_last.review_count IS NOT NULL
    )
    SELECT
        p.product_id,
        p.price_first,
        p.price_last,
        ROUND(100.0 * (p.price_last - p.price_first) / p.price_first, 2) AS pct_change,
        p.review_count
    FROM prices p
""", conn)

if df_price.empty:
    print("WARNING 7.3: No price movement data — skipping.")
else:
    df_meta = pd.read_sql("""
        SELECT DISTINCT product_id, product_name, tier_canonical
        FROM products_weekly_full
    """, conn)
    df_price = df_price.merge(df_meta, on='product_id', how='left')
    df_scatter = df_price[
        (df_price['pct_change'].abs() > 5) &
        (df_price['review_count'] > 50) &
        (df_price['tier_canonical'].isin([
            'skincare','mass_skincare','sensitive_skincare',
            'prestige_skincare','sun_protection','dermo_skincare',
            'cosmetics','mass_cosmetics','prestige_cosmetics','base_makeup'
        ]))
    ].copy()

    if df_scatter.empty:
        print("WARNING 7.3: No products pass the pct_change > 5% AND review_count > 50 filter — skipping.")
    else:
        df_scatter['colour'] = df_scatter['tier_canonical'].apply(tier_colour)
        df_scatter['hover_name'] = df_scatter['product_name'].str[:30]

        fig = go.Figure()
        fig.add_vline(x=0, line_color='#E0E0E0', line_width=1)
        fig.add_trace(go.Scatter(
            x=df_scatter['pct_change'],
            y=df_scatter['review_count'],
            mode='markers',
            marker=dict(
                color=df_scatter['colour'].tolist(),
                size=8,
                opacity=0.7,
            ),
            customdata=df_scatter[['hover_name', 'price_first', 'price_last', 'pct_change', 'review_count']].values,
            hovertemplate=(
                "<b>%{customdata[0]}</b><br>"
                "Price before: ¥%{customdata[1]:,.0f}<br>"
                "Price now: ¥%{customdata[2]:,.0f}<br>"
                "Change: %{customdata[3]:.1f}%<br>"
                "Reviews: %{customdata[4]:,}<extra></extra>"
            ),
        ))
        fig.update_layout(**LAYOUT_DEFAULTS, title="Price Movement vs Consumer Engagement", height=500)
        fig.update_xaxes(showgrid=False, title="Price change % (first → latest snapshot)")
        fig.update_yaxes(gridcolor='#E0E0E0', title="Review count (log scale)", type="log")
        fig.show()
        print("Chart 3 ✓")

# ──────────────────────────────────────────────────────────────────────────
# ## 7.4 New Product Arrivals by Snapshot Date (Grouped Bar)
# ──────────────────────────────────────────────────────────────────────────
df_new = pd.read_sql(f"""
    SELECT snapshot_date, tier_canonical, COUNT(*) AS new_count
    FROM products_weekly_full
    WHERE is_new_product = 1
      AND tier_canonical IN ({SKIN_COS_TIERS})
    GROUP BY snapshot_date, tier_canonical
    ORDER BY snapshot_date, tier_canonical
""", conn)

if df_new.empty:
    print("WARNING 7.4: No new product arrival data — skipping.")
else:
    tiers = df_new['tier_canonical'].unique().tolist()
    fig = go.Figure()
    for tier in tiers:
        sub = df_new[df_new['tier_canonical'] == tier]
        fig.add_trace(go.Bar(
            x=sub['snapshot_date'],
            y=sub['new_count'],
            name=tier,
            marker_color=tier_colour(tier),
            hovertemplate=(
                f"<b>{tier}</b><br>"
                "Date: %{x}<br>New products: %{y}<extra></extra>"
            ),
        ))
    fig.update_layout(**LAYOUT_DEFAULTS, title="New Product Arrivals Per Snapshot", barmode='group', height=400)
    fig.update_xaxes(showgrid=False, title="Snapshot date")
    fig.update_yaxes(gridcolor='#E0E0E0', title="New products ingested")
    fig.show()
    print("Chart 4 ✓")

# ── Summary ────────────────────────────────────────────────────────────────
n_snapshots = len(snapshot_dates)
n_ranking = pd.read_sql(
    "SELECT COUNT(DISTINCT product_id) FROM products_weekly WHERE is_ranking = 1", conn
).iloc[0, 0]
n_price_changed = pd.read_sql("""
    SELECT COUNT(DISTINCT product_id) FROM (
        SELECT product_id, MIN(price_jpy) AS mn, MAX(price_jpy) AS mx
        FROM products_weekly WHERE price_jpy IS NOT NULL GROUP BY product_id
        HAVING mn != mx
    )
""", conn).iloc[0, 0]

print()
print("─" * 50)
print(f"Section 7 summary")
print(f"  Snapshot dates found   : {n_snapshots}")
print(f"  Products in ranking    : {n_ranking:,}")
print(f"  Products with price Δ  : {n_price_changed:,}")
print("─" * 50)

Snapshot dates found: ['2026-04-02', '2026-04-10', '2026-04-17', '2026-04-25', '2026-05-05', '2026-05-12']


Chart 1 ✓


Chart 2 ✓


Chart 3 ✓


Chart 4 ✓



──────────────────────────────────────────────────
Section 7 summary
  Snapshot dates found   : 6
  Products in ranking    : 679
  Products with price Δ  : 4,534
──────────────────────────────────────────────────


In [12]:
# ── Final summary ─────────────────────────────────────────────────────────
total_pw = conn.execute(
    "SELECT COUNT(*) FROM products_weekly"
).fetchone()[0]
total_dates = conn.execute(
    "SELECT COUNT(DISTINCT snapshot_date) FROM products_weekly"
).fetchone()[0]

conn.close()
print()
print('=' * 60)
print('NB02c COMPLETE')
print('=' * 60)
print(f'  products_weekly rows   : {total_pw:,}')
print(f'  Snapshot dates         : {total_dates}')
print()
print('Weekly workflow:')
print('  1. Run NB01a (Sunday morning, fixed IP connection)')
print('  2. Run NB02c Section 4 (set SNAPSHOT_DATE = today)')
print('  3. Section 6 signals update automatically')
print('  4. NB07 dashboard reflects new data')
print()
print('Signals unlock schedule:')
print('  Week 1  : Baseline established (today)')
print('  Week 2  : Review velocity, ranking movement, price changes')
print('  Week 4  : Monthly trend visible')
print('  Week 12 : Quarterly momentum — seasonality starting to show')
print('  Week 52 : Full annual cycle — the data moat is real')



NB02c COMPLETE
  products_weekly rows   : 174,397
  Snapshot dates         : 6

Weekly workflow:
  1. Run NB01a (Sunday morning, fixed IP connection)
  2. Run NB02c Section 4 (set SNAPSHOT_DATE = today)
  3. Section 6 signals update automatically
  4. NB07 dashboard reflects new data

Signals unlock schedule:
  Week 1  : Baseline established (today)
  Week 2  : Review velocity, ranking movement, price changes
  Week 4  : Monthly trend visible
  Week 12 : Quarterly momentum — seasonality starting to show
  Week 52 : Full annual cycle — the data moat is real
